# 02. 물품보관함 데이터 처리 과정

[Colab 에서 열기](https://colab.research.google.com/github/haechan-kim0901/olpark-onestop-guide/blob/main/notebooks/02_lockers_api.ipynb)

공연 보러 온 관객이 **"짐 맡길 곳이 지금 비어 있나?"** 를 바로 알 수 있도록,
전국 공영 물품보관함 데이터에서 **올림픽공원 주변 보관함의 크기별 빈 칸 수**만 뽑아 웹 화면에 보여 줍니다.
이 노트북은 그 과정을 한 단계씩 나눠서 보여 줍니다.

**실행하지 않아도 읽을 수 있습니다.** 아래 표와 숫자는 미리 실행해서 저장해 둔 결과입니다 (기준 시각은 1단계 출력에 나옵니다).

### 전체 흐름

```
공공데이터포털 API 3개 (전국 보관함)
      │  GitHub Actions 가 15분마다 scripts/update_lockers.py 실행
      ▼
data/raw/lockers/          ← 전처리 전: 받은 그대로 (전국 약 380곳)
      │  ① 거리로 고르기  ② 세 표 합치기  ③ 화면 형식으로 바꾸기
      ▼
docs/data/lockers.json     ← 웹 화면이 읽는 파일
data/processed/lockers_olympicpark.csv  ← 팀 확인용 표 (엑셀로 열림)
```

| 단계 | 하는 일 | 한 줄 설명 |
|---|---|---|
| 0 | 준비 | 모드 고르기, 설정값 |
| 1 | 가져오기 | API 3개(위치, 요금, 실시간 빈 칸) 받기 |
| 2 | 정리 | 1.5km 안만 고르기, 보관함 번호로 합치기, 화면 형식으로 바꾸기 |
| 3 | 안정성 | 오류가 나도 화면이 망가지지 않게 하는 장치, 결과 점검 |
| 4 | 결과 형태 | 화면용 JSON, 확인용 CSV 모양 |

> 실제 자동 갱신은 `scripts/update_lockers.py` 가 합니다. 이 노트북은 같은 계산을 풀어서 보여 주고,
> 3단계에서 **노트북 결과와 스크립트 결과가 똑같은지** 직접 확인합니다.

### 처음 보는 말 풀이

| 말 | 뜻 |
|---|---|
| API | 다른 기관 서버에 "이 자료 주세요" 하고 요청하면 자료를 돌려주는 창구. 식당 주문 창구와 비슷합니다 |
| 인증키 | API 를 쓸 수 있는 개인 비밀번호. **GitHub, 노션, 단톡방에 절대 붙여 넣지 않기** |
| JSON | 프로그램끼리 주고받는 글자 형식. `{"이름": "값"}` 모양 |
| CSV | 쉼표로 칸을 나눈 표 파일. 엑셀로 열립니다 |
| 위도, 경도 | 지도 위 위치를 나타내는 두 숫자 (위도는 남북, 경도는 동서) |
| `stlckId` | 보관함 번호. 세 API 표를 이어 붙이는 열쇠 역할 |

## 0. 준비

`MODE` 하나만 고르면 됩니다.

- `"saved"` (기본): GitHub Actions 가 저장소에 올려 둔 **가장 최근 원본**을 씁니다. 인증키가 필요 없어서 누구나 실행할 수 있습니다.
- `"live"`: 지금 바로 API 를 부릅니다. 인증키가 필요합니다. 환경 변수 `DATA_GO_KR_KEY` 에서 먼저 찾고, 없으면 입력 칸이 뜹니다.

In [1]:
MODE = "saved"       # "saved" 또는 "live"
SAVE_FILES = False   # True 로 바꾸면 4단계 결과를 output/ 폴더에 파일로 저장 (Colab 은 zip 으로 내려받기)
SCRIPT_FILE = "update_lockers.py"

import json, math, os, re, sys, urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
KST = timezone(timedelta(hours=9))

# 저장소 폴더 찾기: 내 컴퓨터에서 notebooks/ 안에서 열었으면 바로 위 폴더가 저장소
ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / "scripts" / SCRIPT_FILE).exists() and (p / "docs" / "index.html").exists()), None)
GITHUB_RAW = "https://raw.githubusercontent.com/haechan-kim0901/olpark-onestop-guide/main"

def read_repo_text(rel):
    """저장소 파일 읽기. 내 컴퓨터에 저장소가 있으면 그 파일, 없으면(Colab) GitHub 에서 최신 파일을 받음."""
    if ROOT:
        return (ROOT / rel).read_text(encoding="utf-8-sig")
    with urllib.request.urlopen(f"{GITHUB_RAW}/{rel}", timeout=30) as r:
        return r.read().decode("utf-8-sig")

# 자동 갱신 스크립트를 불러옴 (3단계에서 "노트북 결과 = 스크립트 결과" 를 확인할 때 씀)
if ROOT:
    sys.path.insert(0, str(ROOT / "scripts"))
else:
    os.makedirs("repo_scripts", exist_ok=True)   # Colab: 스크립트를 GitHub 에서 받아 이 폴더에 둠
    for f in ("csv_out.py", SCRIPT_FILE):
        Path("repo_scripts", f).write_text(read_repo_text(f"scripts/{f}"), encoding="utf-8")
    sys.path.insert(0, "repo_scripts")
import importlib
script = importlib.import_module(SCRIPT_FILE[:-3])

print("파일 읽는 곳:", "내 컴퓨터의 저장소 폴더" if ROOT else "GitHub 저장소 (Colab 등)")
print("모드:", MODE)

파일 읽는 곳: 내 컴퓨터의 저장소 폴더
모드: saved


In [2]:
# 스크립트와 같은 설정값
CENTER_LAT, CENTER_LON = 37.5207, 127.1214   # 올림픽공원 중심 (평화의광장 부근)
RADIUS_M = 1500                              # 중심에서 1.5km 안 보관함만 사용 (팀 표준 범위)
VENUES = {                                   # 공연장 좌표 (팀 표준 이름)
    "dome":  (37.5196, 127.1273),   # KSPO DOME
    "hall":  (37.5180, 127.1250),   # 올림픽홀
    "arena": (37.5186, 127.1300),   # 핸드볼경기장
    "lawn":  (37.5213, 127.1196),   # 88잔디마당
}
VENUE_KO = {"dome": "KSPO DOME", "hall": "올림픽홀", "arena": "핸드볼경기장", "lawn": "88잔디마당"}
NEAR_M = 600                                 # 공연장에서 600m 안이면 "가까운 공연장" 으로 표시

# 노트북 설정값이 스크립트와 다르면 바로 알려 줌
assert (CENTER_LAT, CENTER_LON, RADIUS_M, VENUES, NEAR_M) == (
    script.CENTER_LAT, script.CENTER_LON, script.RADIUS_M, script.VENUES, script.NEAR_M), "설정값이 스크립트와 다릅니다"
print("설정값이 scripts/update_lockers.py 와 같습니다.")

설정값이 scripts/update_lockers.py 와 같습니다.


In [3]:
# live 모드일 때만 인증키를 씀. 키는 코드, 파일, 출력 어디에도 남기지 않음
KEY = None
if MODE == "live":
    from urllib.parse import unquote
    KEY = os.environ.get("DATA_GO_KR_KEY", "").strip()   # 1순위: 환경 변수
    if not KEY:                                           # 2순위: 직접 입력 (화면에 안 보임)
        from getpass import getpass
        KEY = getpass("공공데이터포털 인증키를 붙여 넣고 Enter: ").strip()
    KEY = unquote(KEY)   # Encoding 키를 넣어도 Decoding 키로 바꿔 줌
    print("키 길이:", len(KEY), "(키 내용은 보여 주지 않습니다)")
else:
    print("saved 모드: 인증키 없이 저장소에 저장된 최근 원본을 씁니다.")

saved 모드: 인증키 없이 저장소에 저장된 최근 원본을 씁니다.


## 1. 가져오기

같은 기관(행정안전부 한국지역정보개발원)의 API 3개를 부릅니다. 주소는 모두 `https://apis.data.go.kr/B551982/psl_v2/` 로 시작합니다.

| API | 알려 주는 것 | 이 프로젝트에서 쓰는 열 |
|---|---|---|
| `locker_info_v2` | 보관함 위치, 이름, 운영 시간 | `stlckId` 번호, `stlckRprsPstnNm` 이름, `stlckDtlPstnNm` 세부 위치, `lat` 위도, `lot` 경도, `wkdyOperBgngTm`/`wkdyOperEndTm` 운영 시간 |
| `locker_detail_info_v2` | 크기, 요금 설명 | `utztnCrgExpln` 요금 설명 |
| `locker_realtime_use_v2` | **지금 크기별 빈 칸 수** | `usePsbltySmlsz/Mdmsz/LrgszStlckCnt` 소형, 중형, 대형 빈 칸, `totDt` 기준 시각 |

요청할 때 보내는 값: `serviceKey`(인증키), `type=JSON`, `pageNo`(쪽 번호), `numOfRows`(한 번에 받을 줄 수, 1000 이하).
정상이면 결과 코드가 `K0` 입니다.

In [4]:
APIS = [("locker_info", "locker_info_v2"), ("locker_detail", "locker_detail_info_v2"), ("locker_realtime", "locker_realtime_use_v2")]
raw = {}
if MODE == "live":
    for name, path in APIS:
        items, _ = script.call(KEY, path)   # 스크립트의 호출 함수를 그대로 씀 (여러 쪽이면 끝까지, K3 이면 재시도)
        raw[name] = {"api": path, "fetched_at": datetime.now(KST).strftime("%Y-%m-%d %H:%M"), "count": len(items), "items": items}
else:
    for name, path in APIS:
        raw[name] = json.loads(read_repo_text(f"data/raw/lockers/{name}.json"))

info, detail, realtime = (raw[n]["items"] for n, _ in APIS)
for n, _ in APIS:
    print(f"{n:16s} {raw[n]['count']:>4}줄   기준 시각 {raw[n]['fetched_at']}")

locker_info       386줄   기준 시각 2026-11-25 17:10
locker_detail     780줄   기준 시각 2026-11-25 17:10
locker_realtime   361줄   기준 시각 2026-11-25 17:10


### 받은 원본 모양 보기
API 가 돌려준 값은 **한 줄 = 보관함 한 곳** 인 표입니다. 숫자도 모두 글자(`"12"`)로 옵니다. 앞 3줄만 봅니다.

In [5]:
df_info = pd.DataFrame(info)
df_detail = pd.DataFrame(detail)
df_rt = pd.DataFrame(realtime)
print("위치 표 열:", len(df_info.columns), "개 /  요금 표 열:", len(df_detail.columns), "개 /  실시간 표 열:", len(df_rt.columns), "개")
df_info[["stlckId", "stlckRprsPstnNm", "stlckDtlPstnNm", "sggNm", "lat", "lot", "wkdyOperBgngTm", "wkdyOperEndTm"]].head(3)

위치 표 열: 24 개 /  요금 표 열: 17 개 /  실시간 표 열: 7 개


,stlckId,stlckRprsPstnNm,stlckDtlPstnNm,sggNm,lat,lot,wkdyOperBgngTm,wkdyOperEndTm
0,A042A,서울식물원 서울식물원(1~12),방문자센터 내부,강서구,37.5663160000,126.8295810000,050000,010000
1,A042B,서울식물원 서울식물원(13~26),온실관 1층 매표소 옆 엘리베이터,강서구,37.5693290000,126.8356250000,050000,010000
2,A042C,서울식물원 서울식물원(27~40),온실관 1층 매표소 옆 엘리베이터,강서구,37.5692730000,126.8356320000,050000,010000


In [6]:
df_rt.head(3)

,stdgCd,lclgvNm,stlckId,usePsbltyLrgszStlckCnt,usePsbltyMdmszStlckCnt,usePsbltySmlszStlckCnt,totDt
0,1100000000,서울특별시,A042A,3,0,9,20261125171004
1,1100000000,서울특별시,A042B,1,4,8,20261125171004
2,1100000000,서울특별시,A042C,1,4,7,20261125171004


In [7]:
print("지역별 보관함 수 (위치 표 기준)")
df_info["sggNm"].value_counts().head(10)

지역별 보관함 수 (위치 표 기준)


sggNm
중구     35
송파구    31
종로구    30
강남구    28
마포구    27
성동구    16
강서구    15
강동구    14
광진구    14
서초구    14
Name: count, dtype: int64

## 2. 정리

### 2-1. 올림픽공원 1.5km 안 보관함만 고르기

전국 보관함 중 관객이 실제로 쓸 만한 곳만 남깁니다.
위도, 경도 두 점 사이 거리는 **하버사인 공식**(지구가 둥근 것을 감안해 거리를 재는 공식)으로 계산합니다.
위도나 경도가 비어 있거나 숫자가 아닌 줄은 거리를 잴 수 없어서 뺍니다.

In [8]:
def dist_m(a_lat, a_lon, b_lat, b_lon):
    """두 지점 사이 거리(미터). 하버사인 공식"""
    p1, p2 = math.radians(a_lat), math.radians(b_lat)
    dp, dl = p2 - p1, math.radians(b_lon - a_lon)
    h = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371000 * math.asin(math.sqrt(h))

def num(v):
    """글자를 숫자로. 숫자가 아니면 None(빈 값)"""
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

near_rows, no_xy = [], 0
for r in info:
    lat, lon = num(r.get("lat")), num(r.get("lot"))
    if lat is None or lon is None:
        no_xy += 1                                  # 좌표가 없으면 거리를 못 재서 뺌
        continue
    d = round(dist_m(CENTER_LAT, CENTER_LON, lat, lon))
    if d <= RADIUS_M:
        near_rows.append((d, r, lat, lon))
near_rows.sort(key=lambda x: x[0])                  # 가까운 순서

nearby = pd.DataFrame([{**r, "위도": lat, "경도": lon, "공원중심거리_m": d} for d, r, lat, lon in near_rows])
print(f"전국 {len(info)}곳 → 좌표 없는 곳 {no_xy}곳 빼기 → 1.5km 안 {len(nearby)}곳")
nearby[["stlckId", "stlckRprsPstnNm", "stlckDtlPstnNm", "위도", "경도", "공원중심거리_m"]]

전국 386곳 → 좌표 없는 곳 0곳 빼기 → 1.5km 안 7곳


,stlckId,stlckRprsPstnNm,stlckDtlPstnNm,위도,경도,공원중심거리_m
0,TL935,9호선 한성백제,2번 출입구 방향 복도,37.516514,127.116245,651
1,TL813,8호선 몽촌토성,몽촌토성역 B1층 1~5번 출구 방면 (편의점(7-ELEVEN) 주변),37.517664,127.112753,834
2,TL550C,9호선 올림픽공원(55~106),3번 출입구 상가 주변,37.516626,127.130734,940
3,TL550A,5호선 올림픽공원(1~54),1~2번 출구 주변,37.516417,127.131092,979
4,TL812,8호선 강동구청,강동구청역 B1층 1~5번 출구 방면 (편의점(7-ELEVEN) 주변),37.530621,127.120582,1106
5,TL934,9호선 송파나루,지하1층 상가 맞은편,37.510494,127.112215,1394
6,TLP551,5호선 방이,방이역 B1층 1번 출구 방면 (판매기실 옆),37.508659,127.126079,1401


### 2-2. 보관함 번호(`stlckId`)로 세 표 합치기

위치 표, 요금 표, 실시간 표는 따로 옵니다. 엑셀의 VLOOKUP 처럼 **같은 보관함 번호끼리 이어 붙입니다.**

- 실시간 표에 없는 보관함은 **뺍니다.** 빈 칸 수를 모르는데 화면에 0으로 보이면 "만석" 으로 오해할 수 있기 때문입니다.
- 요금 표는 한 보관함에 여러 줄이 있을 수 있어서, 스크립트와 똑같이 **번호마다 마지막 줄**을 씁니다.

In [9]:
rt_by_id = {str(r.get("stlckId")): r for r in realtime}   # 번호 → 실시간 한 줄
dt_by_id = {str(r.get("stlckId")): r for r in detail}     # 번호 → 요금 한 줄 (같은 번호면 마지막 줄)

def cnt(v):
    """빈 칸 수를 정수로. 숫자가 아니면 None"""
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

merged = nearby[["stlckId", "stlckRprsPstnNm", "공원중심거리_m"]].copy()
merged["실시간_있음"] = merged["stlckId"].astype(str).map(lambda s: s in rt_by_id)
for ko, field in (("소형_빈칸", "usePsbltySmlszStlckCnt"), ("중형_빈칸", "usePsbltyMdmszStlckCnt"), ("대형_빈칸", "usePsbltyLrgszStlckCnt")):
    merged[ko] = merged["stlckId"].astype(str).map(lambda s: cnt(rt_by_id.get(s, {}).get(field)))
merged["실시간_기준"] = merged["stlckId"].astype(str).map(lambda s: rt_by_id.get(s, {}).get("totDt", ""))
merged["요금_있음"] = merged["stlckId"].astype(str).map(lambda s: bool(dt_by_id.get(s, {}).get("utztnCrgExpln")))

print(f"1.5km 안 {len(merged)}곳 중 실시간 값 있는 곳 {int(merged['실시간_있음'].sum())}곳 (없는 곳은 화면에서 뺌)")
merged

1.5km 안 7곳 중 실시간 값 있는 곳 7곳 (없는 곳은 화면에서 뺌)


,stlckId,stlckRprsPstnNm,공원중심거리_m,실시간_있음,소형_빈칸,중형_빈칸,대형_빈칸,실시간_기준,요금_있음
0,TL935,9호선 한성백제,651,True,6,2,1,20261125171004,True
1,TL813,8호선 몽촌토성,834,True,5,2,1,20261125171003,True
2,TL550C,9호선 올림픽공원(55~106),940,True,33,8,8,20261125171004,True
3,TL550A,5호선 올림픽공원(1~54),979,True,39,4,11,20261125171004,True
4,TL812,8호선 강동구청,1106,True,6,2,1,20261125171003,True
5,TL934,9호선 송파나루,1394,True,6,2,0,20261125171004,True
6,TLP551,5호선 방이,1401,True,6,2,1,20261125171004,True


### 2-3. 화면 형식으로 바꾸기

웹 화면(`docs/index.html`)이 바로 쓸 수 있게 값을 다듬습니다.

| 원본 | 바꾼 값 | 이유 |
|---|---|---|
| 운영 시간 `0500`, `0100` | `05:00~01:00` | 사람이 읽기 쉽게 |
| 빈 칸 `"9"` (글자) | `9` (숫자) | 계산, 색 표시(0칸 만석 등)에 쓰려고 |
| 크기별 전체 칸 수 | 비움 (`null`) | API 가 전체 칸 수는 주지 않음 |
| 공연장까지 거리 | `near` 목록 | 600m 안 공연장 이름 (KSPO DOME 등) |
| 이름 `9호선 한성백제` | 영어 이름 `Hanseong Baekje Stn (Line 9)` | 다국어 화면용. API 에 영어가 없어 역 이름 표로 만듦 |
| 위도, 경도 | 지도 위치 `x`, `y` | 데모 지도가 실제 축척이 아닌 그림 지도라서 위치를 맞춰 줌 |
| 실시간 `totDt` `20261125171004` | `2026-11-25 17:10` | 팀 표준 시각 형식 (KST) |

지도 위치 계산(`place_of`)은 그림 지도에만 필요한 부분이라 스크립트 함수를 그대로 불러 씁니다.

In [10]:
def hhmm(v):
    v = str(v or "")
    return f"{v[:2]}:{v[2:4]}" if len(v) >= 4 else ""

lockers, latest = [], ""
for d, r, lat, lon in near_rows:
    sid = str(r.get("stlckId"))
    rt = rt_by_id.get(sid)
    if not rt:
        continue                                   # 실시간 값 없으면 뺌
    sizes = {}
    for key, field in (("S", "usePsbltySmlszStlckCnt"), ("M", "usePsbltyMdmszStlckCnt"), ("L", "usePsbltyLrgszStlckCnt")):
        e = cnt(rt.get(field))
        if e is not None:
            sizes[key] = {"total": None, "empty": e}
    if not sizes:
        continue                                   # 크기별 빈 칸이 하나도 숫자가 아니면 뺌
    latest = max(latest, str(rt.get("totDt") or ""))
    name = str(r.get("stlckRprsPstnNm") or sid)
    x, y, name_en = script.place_of(name, lat, lon)   # 그림 지도 위치, 영어 이름
    spot = str(r.get("stlckDtlPstnNm") or "")
    open_, close = hhmm(r.get("wkdyOperBgngTm")), hhmm(r.get("wkdyOperEndTm"))
    lockers.append({
        "id": sid, "x": x, "y": y, "lat": lat, "lon": lon, "dist_m": d,
        "near": [v for v, (vl, vo) in VENUES.items() if dist_m(lat, lon, vl, vo) <= NEAR_M],
        "op": True,
        "hours": f"{open_}~{close}" if open_ else "",
        "name": {"ko": name, "en": name_en},
        "spot": {"ko": spot, "en": spot},
        "fee": str(dt_by_id.get(sid, {}).get("utztnCrgExpln") or ""),
        "sizes": sizes,
    })

seen = {}
for k in lockers:            # 같은 역 보관함이 여러 개면 지도에서 겹치지 않게 옆으로 조금씩
    n = seen.get((k["x"], k["y"]), 0)
    seen[(k["x"], k["y"])] = n + 1
    if n:
        k["x"] = max(14, k["x"] - 22 * n)

stamp = datetime.strptime(latest[:12], "%Y%m%d%H%M") if len(latest) >= 12 else datetime.now(KST).replace(tzinfo=None)
out = {
    "sample": False,
    "source": "행정안전부 한국지역정보개발원_(전국 통합데이터) 공영 물품보관함 현황 실시간 정보 (공공데이터포털)",
    "fetched_at": stamp.strftime("%Y-%m-%d %H:%M"),
    "updated": stamp.strftime("%H:%M"),
    "lockers": lockers,
}
print(f"화면에 보여 줄 보관함 {len(lockers)}곳, 기준 시각 {out['fetched_at']}")
print()
print("첫 번째 보관함 (요금 설명은 길어서 앞부분만):")
first = dict(lockers[0], fee=lockers[0]["fee"][:40] + "...")
print(json.dumps(first, ensure_ascii=False, indent=1))

화면에 보여 줄 보관함 7곳, 기준 시각 2026-11-25 17:10

첫 번째 보관함 (요금 설명은 길어서 앞부분만):
{
 "id": "TL935",
 "x": 126,
 "y": 270,
 "lat": 37.516514,
 "lon": 127.1162454,
 "dist_m": 651,
 "near": [],
 "op": true,
 "hours": "05:00~01:00",
 "name": {
  "ko": "9호선 한성백제",
  "en": "Hanseong Baekje Stn (Line 9)"
 },
 "spot": {
  "ko": "2번 출입구 방향 복도",
  "en": "2번 출입구 방향 복도"
 },
 "fee": "・ 4시간 기본요금 부과 이후 1시간당 추가요금발생\r\n・ 맡길 때 기본요...",
 "sizes": {
  "S": {
   "total": null,
   "empty": 6
  },
  "M": {
   "total": null,
   "empty": 2
  },
  "L": {
   "total": null,
   "empty": 1
  }
 }
}


## 3. 안정성

자동 갱신은 사람이 지켜보지 않을 때도 15분마다 돕니다. 그래서 **문제가 생겨도 화면이 깨지지 않게** 아래 장치를 넣었습니다.

| 이런 일이 생기면 | 스크립트가 하는 일 | 화면에 보이는 것 |
|---|---|---|
| 인증키가 없음 | 바로 멈춤, 기존 `lockers.json` 그대로 | 직전 값 |
| `K3` (조건에 맞는 자료 없음) | 서울 지역 코드(`stdgCd=1100000000`)를 넣어 한 번 더 요청 | 정상 값 |
| 그 밖의 오류 코드, JSON 아닌 응답 | 멈춤, 기존 파일 그대로 | 직전 값 |
| 위치나 실시간 자료가 0건 | 멈춤, 기존 파일 그대로 | 직전 값 |
| 1.5km 안 보관함이 0곳 | 멈춤, 기존 파일 그대로 | 직전 값 |
| 실시간 값이 없는 보관함 | 그 보관함만 뺌 | "만석" 오해 없음 |
| 숫자가 아닌 값 (`""`, `"-"`) | 빈 값(None)으로 처리, 오류로 멈추지 않음 | 해당 크기 안 보임 |
| 자료가 1,000줄 넘음 | 다음 쪽까지 끝까지 받음 | 빠진 보관함 없음 |
| 자료가 전과 같음 | 커밋하지 않음 | 변화 없음 |
| 갱신이 1시간 넘게 멈춤 | (화면) 5분마다 다시 읽고, 1시간 넘으면 빨간 글씨 경고 | 오래된 값이라고 알림 |
| 인증키 | 코드, 원본 JSON, CSV 어디에도 저장 안 함 (GitHub Secret 에만) | 공개 저장소에 키 없음 |

아래 셀은 지금 결과가 믿을 만한지 직접 점검합니다.

In [11]:
checks = []
def check(name, ok, detail=""):
    checks.append({"점검": name, "결과": "통과" if ok else "확인 필요", "내용": detail})

check("보관함이 1곳 이상", len(lockers) > 0, f"{len(lockers)}곳")
n_no_rt = int((~merged["실시간_있음"]).sum())
check("실시간 값 없는 곳은 뺐음", all(k["id"] in rt_by_id for k in lockers), f"뺀 곳 {n_no_rt}곳")
empties = [s["empty"] for k in lockers for s in k["sizes"].values()]
check("빈 칸 수가 0 이상 정수", all(isinstance(e, int) and e >= 0 for e in empties), f"값 {len(empties)}개, 최소 {min(empties)}, 최대 {max(empties)}")
check("모두 1.5km 안", all(k["dist_m"] <= RADIUS_M for k in lockers), f"가장 먼 곳 {max(k['dist_m'] for k in lockers)}m")
check("좌표가 서울 동남권 범위", all(37.4 < k["lat"] < 37.6 and 127.0 < k["lon"] < 127.2 for k in lockers))
check("ID 중복 없음", len({k["id"] for k in lockers}) == len(lockers))
check("시각이 팀 표준 형식 (YYYY-MM-DD HH:MM)", bool(re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}", out["fetched_at"])), out["fetched_at"])
raw_text = "".join(json.dumps(raw[n], ensure_ascii=False) for n, _ in APIS)
check("원본 파일에 인증키 없음", "serviceKey" not in raw_text and (KEY is None or KEY not in raw_text))

age_min = (datetime.now(KST).replace(tzinfo=None) - stamp).total_seconds() / 60
checks.append({"점검": "얼마나 최근 값인가 (실행한 때 기준)", "결과": "통과" if age_min <= 60 else "참고",
               "내용": f"{age_min:.0f}분 전 값" + ("" if age_min <= 60 else ", Actions 예약 실행이 늦으면 생김. 화면은 1시간 넘으면 경고")})
pd.DataFrame(checks)

,점검,결과,내용
0,보관함이 1곳 이상,통과,7곳
1,실시간 값 없는 곳은 뺐음,통과,뺀 곳 0곳
2,빈 칸 수가 0 이상 정수,통과,"값 21개, 최소 0, 최대 39"
3,모두 1.5km 안,통과,가장 먼 곳 1401m
4,좌표가 서울 동남권 범위,통과,
5,ID 중복 없음,통과,
6,시각이 팀 표준 형식 (YYYY-MM-DD HH:MM),통과,2026-11-25 17:10
7,원본 파일에 인증키 없음,통과,
8,얼마나 최근 값인가 (실행한 때 기준),참고,"92분 전 값, Actions 예약 실행이 늦으면 생김. 화면은 1시간 넘으면 경고"


### 노트북 결과 = 자동 갱신 스크립트 결과?

같은 원본을 `scripts/update_lockers.py` 의 `build()` 에 넣어서, 위에서 단계별로 만든 결과와 **한 글자도 다르지 않은지** 비교합니다.
나중에 누가 스크립트를 고쳤는데 노트북은 그대로라면, 여기서 "다름" 이 나와 바로 알 수 있습니다.

In [12]:
by_script = script.build(info, detail, realtime)
by_script.pop("_설명", None)
same = by_script == out
print("스크립트 결과와 같음" if same else "스크립트 결과와 다름: 노트북을 스크립트에 맞게 고쳐야 합니다")
assert same

스크립트 결과와 같음


## 4. 결과 형태

결과는 두 가지 파일로 저장됩니다. 내용은 같고 쓰임새만 다릅니다.

| 파일 | 누가 보나 | 모양 |
|---|---|---|
| `docs/data/lockers.json` | 웹 화면 | 2-3단계 출력처럼 보관함마다 `{...}` 묶음 |
| `data/processed/lockers_olympicpark.csv` | 팀원, 기획서 | 아래 표 (한 줄 = 보관함 한 곳, UTF-8 BOM 이라 엑셀에서 한글이 안 깨짐) |

`lockers.json` 의 주요 항목: `fetched_at` 기준 시각, `lockers[].name` 한/영 이름, `dist_m` 공원 중심 거리, `near` 가까운 공연장,
`sizes.S/M/L.empty` 소형, 중형, 대형 빈 칸, `fee` 요금 설명.

In [13]:
# 확인용 CSV 와 같은 표 (스크립트의 processed_rows 와 같은 함수로 만듦)
processed = pd.DataFrame(script.processed_rows(out), columns=script.PROCESSED_COLS)
processed["요금설명"] = processed["요금설명"].str.slice(0, 30) + "..."   # 화면에서 보기 좋게 앞부분만
processed.drop(columns=["출처"])

,기준시각,보관함ID,보관함이름,세부위치,위도,경도,공원중심거리_m,가까운공연장,운영시간,소형_빈칸,중형_빈칸,대형_빈칸,요금설명
0,2026-11-25 17:10,TL935,9호선 한성백제,2번 출입구 방향 복도,37.516514,127.116245,651,,05:00~01:00,6,2,1,・ 4시간 기본요금 부과 이후 1시간당 추가요금발생 /...
1,2026-11-25 17:10,TL813,8호선 몽촌토성,몽촌토성역 B1층 1~5번 출구 방면 (편의점(7-ELEVEN) 주변),37.517664,127.112753,834,,05:00~01:00,5,2,1,・ 4시간 기본요금 부과 이후 1시간당 추가요금발생 /...
2,2026-11-25 17:10,TL550C,9호선 올림픽공원(55~106),3번 출입구 상가 주변,37.516626,127.130734,940,"KSPO DOME, 올림픽홀, 핸드볼경기장",05:00~01:00,33,8,8,・ 4시간 기본요금 부과 이후 1시간당 추가요금발생 /...
3,2026-11-25 17:10,TL550A,5호선 올림픽공원(1~54),1~2번 출구 주변,37.516417,127.131092,979,"KSPO DOME, 올림픽홀, 핸드볼경기장",05:00~01:00,39,4,11,・ 4시간 기본요금 부과 이후 1시간당 추가요금발생 /...
4,2026-11-25 17:10,TL812,8호선 강동구청,강동구청역 B1층 1~5번 출구 방면 (편의점(7-ELEVEN) 주변),37.530621,127.120582,1106,,05:00~01:00,6,2,1,・ 4시간 기본요금 부과 이후 1시간당 추가요금발생 /...
5,2026-11-25 17:10,TL934,9호선 송파나루,지하1층 상가 맞은편,37.510494,127.112215,1394,,05:00~01:00,6,2,0,・ 4시간 기본요금 부과 이후 1시간당 추가요금발생 /...
6,2026-11-25 17:10,TLP551,5호선 방이,방이역 B1층 1번 출구 방면 (판매기실 옆),37.508659,127.126079,1401,,05:00~01:00,6,2,1,・ 4시간 기본요금 부과 이후 1시간당 추가요금발생 /...


In [14]:
# 저장소에 올라가 있는 결과 CSV 와 비교 (saved 모드일 때 같아야 정상)
if MODE == "saved":
    import io
    repo_csv = pd.read_csv(io.StringIO(read_repo_text("data/processed/lockers_olympicpark.csv")), dtype=str).fillna("")
    mine = pd.DataFrame(script.processed_rows(out), columns=script.PROCESSED_COLS).astype(str).replace({"None": ""})
    if len(repo_csv) == len(mine) and (repo_csv.values == mine.values).all():
        print(f"저장소의 data/processed/lockers_olympicpark.csv 와 같습니다 ({len(mine)}줄).")
    else:
        print("저장소 CSV 와 다릅니다. 노트북을 연 사이 Actions 가 새 값을 올렸을 수 있습니다. 다시 실행해 보세요.")

저장소의 data/processed/lockers_olympicpark.csv 와 같습니다 (7줄).


In [15]:
# SAVE_FILES = True 일 때만: 결과를 output/ 폴더에 저장 (Colab 이면 zip 으로 내려받기)
if SAVE_FILES:
    import csv, shutil
    os.makedirs("output", exist_ok=True)
    with open("output/lockers.json", "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1)
    with open("output/lockers_olympicpark.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=script.PROCESSED_COLS)
        w.writeheader()
        w.writerows({k: ("" if v is None else v) for k, v in r.items()} for r in script.processed_rows(out))
    shutil.make_archive("lockers_output", "zip", "output")
    try:
        from google.colab import files
        files.download("lockers_output.zip")
    except ImportError:
        print("output/ 폴더와 lockers_output.zip 에 저장했습니다.")
else:
    print("파일 저장은 건너뜀 (SAVE_FILES = False). 저장소 파일은 Actions 가 15분마다 자동으로 바꿉니다.")

파일 저장은 건너뜀 (SAVE_FILES = False). 저장소 파일은 Actions 가 15분마다 자동으로 바꿉니다.


## 기획서에 쓸 수 있는 설명 (예시)

> 행정안전부 한국지역정보개발원의 공영 물품보관함 실시간 API 3종(위치, 상세, 실시간 이용)을 15분마다 자동 수집합니다.
> 전국 약 380곳 중 올림픽공원 중심 반경 1.5km 안 보관함을 위도, 경도 거리 계산으로 고르고, 보관함 번호로 세 자료를 결합했습니다.
> 실시간 값이 없는 보관함은 만석 오해를 막기 위해 제외하고, 크기별 빈 칸 수, 가까운 공연장(600m 이내), 운영 시간, 요금을 화면용 형식으로 변환했습니다.
> 수집이 실패하면 직전 값을 유지하고, 1시간 넘게 갱신되지 않으면 화면에 경고를 띄웁니다.